# Demo 6: Spectral Laplacian Family on a 3D Box Domain

This notebook introduces the **spectral Laplacian family** in `pygeoinf3d`:

| Class | Formula | Role |
|---|---|---|
| `BoxLaplacian` | $L = -\Delta$ | Negative Laplacian (positive definite) |
| `InverseBoxLaplacian` | $L^{-1} = (-\Delta)^{-1}$ | Pseudo-inverse |
| `GeneralizedBoxLaplacian` | $(-\alpha\Delta)^s$ | Parameterised power |
| `KLSampler3D` | $f \sim \mathcal{N}(0, C)$ | Karhunen–Loève sampler |

All operators act **spectrally**: given eigenfunctions $\Phi_k$ and eigenvalues $\lambda_k$ of $-\Delta$ (Dirichlet BCs),
$$
((-\alpha\Delta)^s f)(\mathbf{x}) = \sum_{k=0}^{N-1} (\alpha\lambda_k)^s \langle f, \Phi_k \rangle \Phi_k(\mathbf{x})
$$

See also:
- `generalized_laplacian_covariance_demo.ipynb` — detailed covariance analysis
- `kl_sampler_prior_demo.ipynb` — KL sampling in depth

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pygeoinf3d import BoxDomain, Lebesgue3D, TensorProductProvider
from pygeoinf3d import BoxLaplacian, InverseBoxLaplacian, GeneralizedBoxLaplacian, KLSampler3D
from pygeoinf3d.plot import plot_function3d_axis_slice

In [ ]:
# Unit-cube domain and Lebesgue space
box = BoxDomain(np.zeros(3), np.ones(3))
space = Lebesgue3D(box)

# Two providers: truncation=5 (125 modes) for spectral analysis,
#                truncation=4 (64 modes)  for sampling (faster)
prov5 = TensorProductProvider(space, truncation=5)
prov4 = TensorProductProvider(space, truncation=4)

print(f"prov5: {prov5.dim} basis functions  (truncation=5)")
print(f"prov4: {prov4.dim} basis functions  (truncation=4)")

## Section 1: BoxLaplacian — Eigenvalue Spectrum

`BoxLaplacian(space, provider)` implements $L = -\Delta$ spectrally on the basis provided by `TensorProductProvider`.

For Dirichlet BCs on the unit cube $[0,1]^3$, the eigenfunctions are
$$\Phi_{n_1,n_2,n_3}(\mathbf{x}) = 2^{3/2}\sin(n_1\pi x_1)\sin(n_2\pi x_2)\sin(n_3\pi x_3), \quad n_i \ge 1$$
with eigenvalues
$$\lambda_{n_1,n_2,n_3} = \pi^2(n_1^2 + n_2^2 + n_3^2)$$

**Weyl's law** predicts the asymptotic eigenvalue distribution: the $n$-th eigenvalue scales like
$$\lambda_n \sim (6\pi^2 n)^{2/3}$$

In [ ]:
# Create BoxLaplacian and extract eigenvalue spectrum
lap = BoxLaplacian(space, prov5)

eigenvalues = np.array([prov5.get_eigenvalue(i) for i in range(prov5.dim)])
eigenvalues_sorted = np.sort(eigenvalues)
indices = np.arange(1, len(eigenvalues_sorted) + 1)

# Weyl's law: lambda_n ~ (6*pi^2 * n)^{2/3}  (unit cube, V=1)
weyl_lambda = (6.0 * np.pi**2 * indices) ** (2.0 / 3.0)

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(indices, eigenvalues_sorted, 'o', ms=4, label='Numerical eigenvalues')
ax.semilogy(indices, weyl_lambda, '--', color='red', lw=1.5,
            label=r"Weyl's law: $(6\pi^2 n)^{2/3}$")
ax.set_xlabel('Mode index $n$')
ax.set_ylabel(r'Eigenvalue $\lambda_n$')
ax.set_title(r'Spectrum of $-\Delta$ (Dirichlet BC, unit cube, $N=125$ modes)')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

lam_min_expected = np.pi**2 * 3
print(f"Smallest eigenvalue: {eigenvalues_sorted[0]:.6f}  (expected pi^2 x 3 = {lam_min_expected:.6f})")
print(f"Largest eigenvalue:  {eigenvalues_sorted[-1]:.4f}")

## Section 2: Eigenfunction Verification

`BoxLaplacian` acts on eigenfunctions as
$$L\Phi_k = \lambda_k\Phi_k$$

We verify this numerically by evaluating both sides at random points in $(0,1)^3$.

In [ ]:
# Verify L*phi_i = lambda_i * phi_i at random interior points
rng_check = np.random.default_rng(2024)
test_pts = rng_check.uniform(0.05, 0.95, (20, 3))

print(f"{'Mode':>4}  {'lambda_i':>12}  {'Max rel. error':>15}")
print("-" * 38)
for i in [0, 1, 3, 5, 10, 20, 50]:
    phi_i = prov5.get_eigenfunction(i)
    lam_i = prov5.get_eigenvalue(i)
    lap_phi_i = lap(phi_i)
    lhs = np.array([lap_phi_i(p) for p in test_pts])
    rhs = np.array([lam_i * phi_i(p) for p in test_pts])
    denom = np.max(np.abs(rhs)) + 1e-15
    err = np.max(np.abs(lhs - rhs)) / denom
    print(f"  {i:3d}  {lam_i:12.4f}  {err:15.2e}")

## Section 3: InverseBoxLaplacian

`InverseBoxLaplacian(space, provider)` implements $L^{-1} = (-\Delta)^{-1}$ — the pseudo-inverse of the Laplacian. Its eigenvalues are $1/\lambda_k$.

We verify the inverse relationship: $L^{-1} \circ L = I$ (applied to a single eigenfunction).

In [ ]:
inv_lap = InverseBoxLaplacian(space, prov5)

# --- Plot 1: 1/lambda_i vs lambda_i ---
inv_evals_expected = 1.0 / eigenvalues_sorted

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax1 = axes[0]
ax1.loglog(eigenvalues_sorted, inv_evals_expected, 'o', ms=4, color='steelblue')
lam_range = np.linspace(eigenvalues_sorted[0], eigenvalues_sorted[-1], 300)
ax1.loglog(lam_range, 1.0 / lam_range, '--', color='red', lw=1.5, label=r'$1/\lambda$')
ax1.set_xlabel(r'$\lambda_n$')
ax1.set_ylabel(r'$\lambda_n^{-1}$')
ax1.set_title(r'Eigenvalues of $(-\Delta)^{-1}$ (log-log)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# --- Plot 2: verify L^{-1} circ L = I on eigenfunction 5 ---
phi5 = prov5.get_eigenfunction(5)
Linv_L_phi5 = inv_lap(lap(phi5))

pts_v = rng_check.uniform(0.05, 0.95, (40, 3))
orig = np.array([phi5(p) for p in pts_v])
recovered = np.array([Linv_L_phi5(p) for p in pts_v])

ax2 = axes[1]
ax2.scatter(orig, recovered, s=20, color='steelblue', alpha=0.7)
lims = [min(orig.min(), recovered.min()) * 1.05, max(orig.max(), recovered.max()) * 1.05]
ax2.plot(lims, lims, 'r--', lw=1.5, label='y = x')
ax2.set_xlabel(r'$\Phi_5(x)$')
ax2.set_ylabel(r'$(L^{-1} o L) Phi_5(x)$')
ax2.set_title(r'Inverse check: $L^{-1} o L = I$')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('InverseBoxLaplacian verification', fontsize=13)
plt.tight_layout()
plt.show()

err = np.max(np.abs(orig - recovered)) / (np.max(np.abs(orig)) + 1e-15)
print(f"Max relative error  |L^-1(L Phi_5) - Phi_5| / |Phi_5|_max = {err:.2e}")

## Section 4: GeneralizedBoxLaplacian — Varying alpha

`GeneralizedBoxLaplacian(space, provider, power=s, alpha=alpha)` implements $(-\alpha\Delta)^s$.

The eigenvalue on mode $k$ is $(\alpha\lambda_k)^s$.

For $s = -1$ (covariance model), the eigenvalue is $(\alpha\lambda_k)^{-1} = 1/(\alpha\lambda_k)$:
- **Larger alpha**: smaller eigenvalues → narrower prior variance → more regularisation
- **Smaller alpha**: larger eigenvalues → broader prior variance → less regularisation

In [ ]:
alphas = [0.1, 0.5, 1.0]
power_sec4 = -1
colors = ['steelblue', 'darkorange', 'green']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Left: eigenvalue spectra (descending rank) ---
ax1 = axes[0]
for alpha, color in zip(alphas, colors):
    gen = GeneralizedBoxLaplacian(space, prov5, power=power_sec4, alpha=alpha)
    evals = np.array([gen.get_eigenvalue(i) for i in range(prov5.dim)])
    evals_desc = np.sort(evals)[::-1]
    ax1.semilogy(evals_desc, 'o-', ms=3, lw=1.2, color=color, label=f'alpha={alpha}')

ax1.set_xlabel('Mode rank (descending eigenvalue)')
ax1.set_ylabel(r'Eigenvalue $(\alpha\lambda_k)^{-1}$')
ax1.set_title(r'Spectra of $(-\alpha\Delta)^{-1}$ for three $\alpha$')
ax1.legend()
ax1.grid(True, which='both', alpha=0.3)

# --- Right: formula curves ---
ax2 = axes[1]
lam_range = np.linspace(eigenvalues_sorted[0], eigenvalues_sorted[-1], 300)
for alpha, color in zip(alphas, colors):
    ax2.semilogy(lam_range, (alpha * lam_range) ** (-1), '-', lw=2,
                 color=color, label=f'alpha={alpha}')
ax2.set_xlabel(r'Laplacian eigenvalue $\lambda_k$')
ax2.set_ylabel(r'$(\alpha\lambda_k)^{-1}$')
ax2.set_title(r'Analytical formula: $(\alpha\lambda_k)^{-1}$')
ax2.legend()
ax2.grid(True, which='both', alpha=0.3)

plt.suptitle('alpha variation (s=-1): larger alpha → smaller covariance eigenvalues → more regularisation',
             fontsize=11)
plt.tight_layout()
plt.show()

## Section 5: Prior Samples via KLSampler3D

`KLSampler3D(operator, n_modes=N, rng=...)` draws random samples from the Gaussian process with covariance $C = (-\alpha\Delta)^{-1}$.

The Karhunen-Loeve expansion truncated to $N$ modes gives
$$f(\mathbf{x}) = \sum_{k=0}^{N-1} \xi_k \sqrt{c_k}\, \Phi_k(\mathbf{x}), \quad \xi_k \sim \mathcal{N}(0, 1)$$
where $c_k = (\alpha\lambda_k)^{-1}$ are the covariance eigenvalues.

We visualise samples as $z = 0.5$ slices of the unit cube.

> **Note:** `truncation=4` (64 modes) is used here for faster execution.

In [ ]:
# 4 independent samples from GP with C = (-0.2 Delta)^{-1}
N_MODES = 40   # KL truncation (<= prov4.dim = 64)
ALPHA_DEMO = 0.2

gen_lap_demo = GeneralizedBoxLaplacian(space, prov4, power=-1, alpha=ALPHA_DEMO)
sampler_demo = KLSampler3D(gen_lap_demo, n_modes=N_MODES, rng=np.random.default_rng(42))

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for i, ax in enumerate(axes.flat):
    sample = sampler_demo.sample()
    plot_function3d_axis_slice(sample, axis=2, coord=0.5, ax=ax, show_plot=False,
                               title=f'Sample {i + 1}')

plt.suptitle('Four KL prior samples: f ~ N(0, (-0.2*Delta)^{-1})\n(z = 0.5 slice, N = 40 modes)',
             fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Compare sample roughness for three values of alpha
alphas_cmp = [0.02, 0.2, 2.0]
titles_cmp = ['Smooth (alpha=0.02)', 'Medium (alpha=0.2)', 'Rough (alpha=2.0)']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for alpha, title, ax in zip(alphas_cmp, titles_cmp, axes):
    gen = GeneralizedBoxLaplacian(space, prov4, power=-1, alpha=alpha)
    samp = KLSampler3D(gen, n_modes=N_MODES, rng=np.random.default_rng(42))
    f = samp.sample()
    plot_function3d_axis_slice(f, axis=2, coord=0.5, ax=ax, show_plot=False, title=title)

plt.suptitle('Effect of alpha on sample roughness  (s=-1, N=40 modes)', fontsize=12)
plt.tight_layout()
plt.show()

## Section 6: Power Variation — Spectral Decay Rate

The **power** $s$ controls how fast the covariance eigenvalues decay with mode index:
$$c_k = (\alpha\lambda_k)^s$$

For $s < 0$, larger $|s|$ gives faster decay and **smoother** samples:
- $s = -1$: $c_k \propto \lambda_k^{-1}$ (slow decay, rougher)
- $s = -2$: $c_k \propto \lambda_k^{-2}$ (moderate decay)
- $s = -3$: $c_k \propto \lambda_k^{-3}$ (fast decay, smoother)

> **Note:** Integer powers only — `GeneralizedBoxLaplacian` raises `ValueError` for fractional powers.

In [ ]:
ALPHA_PW = 0.1
powers_cmp = [-1, -2, -3]
colors_pw = ['steelblue', 'darkorange', 'green']

# --- Eigenvalue decay curves ---
fig, ax_spec = plt.subplots(figsize=(8, 5))
for pw, color in zip(powers_cmp, colors_pw):
    evals_pw = (ALPHA_PW * eigenvalues_sorted) ** pw
    ax_spec.semilogy(np.arange(1, len(evals_pw) + 1), np.sort(evals_pw)[::-1],
                     'o-', ms=3, lw=1.2, color=color, label=f's={pw}')

ax_spec.set_xlabel('Mode rank (descending)')
ax_spec.set_ylabel(r'$(\alpha\lambda_k)^s$')
ax_spec.set_title(r'Spectral decay for varying power $s$ ($\alpha=0.1$)')
ax_spec.legend()
ax_spec.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

# --- Sample slices ---
titles_pw = ['s = -1 (roughest)', 's = -2', 's = -3 (smoothest)']
fig2, axes2 = plt.subplots(1, 3, figsize=(14, 4))
for pw, title, ax in zip(powers_cmp, titles_pw, axes2):
    gen_pw = GeneralizedBoxLaplacian(space, prov4, power=pw, alpha=ALPHA_PW)
    samp_pw = KLSampler3D(gen_pw, n_modes=N_MODES, rng=np.random.default_rng(7))
    f_pw = samp_pw.sample()
    plot_function3d_axis_slice(f_pw, axis=2, coord=0.5, ax=ax, show_plot=False, title=title)

plt.suptitle('Effect of power s on sample smoothness  (alpha=0.1, z=0.5 slice)', fontsize=12)
plt.tight_layout()
plt.show()

## Section 7: Connection to Existing Demos

This notebook gives a concise overview. For deeper treatment see:

- **`generalized_laplacian_covariance_demo.ipynb`** — Detailed study of the covariance structure of `GeneralizedBoxLaplacian`: prior covariance kernels, trace decay, cross-sections.

- **`kl_sampler_prior_demo.ipynb`** — In-depth KL sampling: convergence with `n_modes`, comparison with pseudo-random samples, use as a prior in inversion.

Both demos share the same core API (`GeneralizedBoxLaplacian` + `KLSampler3D`) as demonstrated here.

## Section 8: Summary

| Class | Constructor | Eigenvalue on $\Phi_k$ | Typical use |
|---|---|---|---|
| `BoxLaplacian` | `BoxLaplacian(space, prov)` | $\lambda_k$ | Differential operator $-\Delta$ |
| `InverseBoxLaplacian` | `InverseBoxLaplacian(space, prov)` | $1/\lambda_k$ | Pseudo-inverse |
| `GeneralizedBoxLaplacian` | `(space, prov, power=s, alpha=a)` | $(\alpha\lambda_k)^s$ | Regularisation / covariance |
| `KLSampler3D` | `(op, n_modes=N, rng=...)` | — | GP prior sampling |

**Parameter effects:**

| Parameter | Larger value | Smaller value |
|---|---|---|
| `alpha` (s < 0) | More regularisation (smaller covariance) | Less regularisation (larger covariance) |
| `|power|` (s < 0) | Smoother samples (faster spectral decay) | Rougher samples |
| `n_modes` in KLSampler3D | More accurate KL expansion | Faster sampling |